# BÀI THỰC HÀNH TUẦN 2B: PHƯƠNG PHÁP NAÏVE BAYES CHO DỮ LIỆU RỜI RẠC
- **Họ và tên**: Nguyễn Hải Anh  
- **MSSV**: 24001968  
- **Học phần**: Machine Learning

## Ví dụ 1: Multinomial Naïve Bayes (Cài đặt thủ công theo hướng dẫn)

In [11]:
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score
import os

# Đọc dữ liệu
orders_fn = 'multinomial_nb_orders.csv' if os.path.exists('multinomial_nb_orders.csv') else 'multinomial_nb_orders (1).csv'
df = pd.read_csv(orders_fn)
target_feature = 'cuisine'

# Chia dữ liệu theo tỉ lệ train:test = 7:3
Train_size = round(len(df) * 0.7)
df_train = df.iloc[:Train_size].copy()
df_test = df.iloc[Train_size:].copy()

X_Train = df_train.drop(columns=[target_feature])
Y_Train = df_train[target_feature].values
X_Test = df_test.drop(columns=[target_feature]).reset_index(drop=True)
Y_Test = df_test[target_feature].reset_index(drop=True).values

# Mảng lưu số lựa chọn tối đa M ứng với mỗi trường (theo yêu cầu tự giải ở trang 1)
feature_cols = X_Train.columns
num_labels_M = {col: len(df[col].unique()) for col in feature_cols}

# Tính xác suất tiên nghiệm P(k)
classes = np.unique(Y_Train)
priors = {c: np.mean(Y_Train == c) for c in classes}

# Tính xác suất P(x_i | k) với làm mịn Laplace
probs = {}
for col in feature_cols:
    probs[col] = {}
    M = num_labels_M[col]
    unique_vals = df[col].unique()
    for c in classes:
        sub_col = X_Train[col][Y_Train == c]
        N_k = len(sub_col)
        counts = sub_col.value_counts().to_dict()
        probs[col][c] = {v: (counts.get(v, 0) + 1.0) / (N_k + M * 1.0) for v in unique_vals}

def predict_output_label(df_input):
    preds = []
    for _, row in df_input.iterrows():
        scores = []
        for c in classes:
            s = np.log(priors[c])
            for col in feature_cols:
                v = row[col]
                s += np.log(probs[col][c].get(v, 1.0 / (len(probs[col][c]) + 1)))
            scores.append(s)
        preds.append(classes[np.argmax(scores)])
    return np.array(preds)

# Dự đoán trên tập test và in độ chính xác
Y_pred = predict_output_label(X_Test)
print("Độ chính xác mô hình tự cài đặt (Scratch):", accuracy_score(Y_Test, Y_pred))

# In thử dự đoán của một mẫu đầu vào
sample_idx = 3
sample_row = pd.DataFrame([X_Train.iloc[sample_idx]])
print(f"Dự đoán mẫu Train[{sample_idx}]:", predict_output_label(sample_row)[0], f"| Thực tế: {Y_Train[sample_idx]}")


Độ chính xác mô hình tự cài đặt (Scratch): 0.47555555555555556
Dự đoán mẫu Train[3]: italian | Thực tế: italian


## Bài tập 1: So sánh với thư viện Scikit-learn và giải thích sai lệch

In [12]:
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

# 1. Chạy thư viện Sklearn với LabelEncoder
X_Train_le = X_Train.copy()
X_Test_le = X_Test.copy()
le = LabelEncoder()
for col in feature_cols:
    X_Train_le[col] = le.fit_transform(X_Train_le[col])
    X_Test_le[col] = le.transform(X_Test_le[col])

clf_le = MultinomialNB()
clf_le.fit(X_Train_le, Y_Train)
y_pred_le = clf_le.predict(X_Test_le)
print("Độ chính xác Sklearn (LabelEncoder):", accuracy_score(Y_Test, y_pred_le))

# 2. Chạy thư viện Sklearn với OneHotEncoder
ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
X_Train_ohe = ohe.fit_transform(X_Train)
X_Test_ohe = ohe.transform(X_Test)

clf_ohe = MultinomialNB()
clf_ohe.fit(X_Train_ohe, Y_Train)
y_pred_ohe = clf_ohe.predict(X_Test_ohe)
print("Độ chính xác Sklearn (OneHotEncoder):", accuracy_score(Y_Test, y_pred_ohe))

# 3. Giải thích lý do sai lệch theo yêu cầu đề bài
print("\nGiải thích sự sai lệch:")
print("- LabelEncoder chuyển biến danh mục thành các số 0, 1, 2... khiến MultinomialNB hiểu nhầm đây là tần suất xuất hiện và gán thứ tự độ lớn giả tạo, làm giảm độ chính xác (~17.67%).")
print("- Khi dùng OneHotEncoder để mã hóa nhị phân các lựa chọn, kết quả của thư viện đạt 47.56%, hoàn toàn khớp với code tự cài đặt.")


Độ chính xác Sklearn (LabelEncoder): 0.17666666666666667
Độ chính xác Sklearn (OneHotEncoder): 0.47555555555555556

Giải thích sự sai lệch:
- LabelEncoder chuyển biến danh mục thành các số 0, 1, 2... khiến MultinomialNB hiểu nhầm đây là tần suất xuất hiện và gán thứ tự độ lớn giả tạo, làm giảm độ chính xác (~17.67%).
- Khi dùng OneHotEncoder để mã hóa nhị phân các lựa chọn, kết quả của thư viện đạt 47.56%, hoàn toàn khớp với code tự cài đặt.


## Ví dụ 2: Bernoulli Naïve Bayes (Dùng thư viện Scikit-learn)

In [13]:
import numpy as np
import pandas as pd
from sklearn.naive_bayes import BernoulliNB
from sklearn.metrics import accuracy_score

# Đọc dữ liệu triệu chứng bệnh
df_symp = pd.read_csv('bernoulli_nb_symptoms.csv')
target_symp = 'condition'

# Chia tập train:test theo tỉ lệ 6:4
Train_size_s = round(len(df_symp) * 0.6)
df_train_s = df_symp.iloc[:Train_size_s].copy()
df_test_s = df_symp.iloc[Train_size_s:].copy()

X_Train_s = df_train_s.drop(columns=[target_symp])
Y_Train_s = df_train_s[target_symp].values
X_Test_s = df_test_s.drop(columns=[target_symp]).reset_index(drop=True)
Y_Test_s = df_test_s[target_symp].reset_index(drop=True).values

# Huấn luyện bằng thư viện BernoulliNB
bnb = BernoulliNB(alpha=1.0)
bnb.fit(X_Train_s, Y_Train_s)
y_pred_s_sk = bnb.predict(X_Test_s)

print("Độ chính xác Bernoulli NB (Sklearn):", accuracy_score(Y_Test_s, y_pred_s_sk))


Độ chính xác Bernoulli NB (Sklearn): 0.485


## Bài tập 2: Tự xây dựng Bernoulli Naïve Bayes (From Scratch)

In [14]:
import numpy as np
from sklearn.metrics import accuracy_score

# Tính xác suất tiên nghiệm P(k)
classes_s = np.unique(Y_Train_s)
priors_s = {c: np.mean(Y_Train_s == c) for c in classes_s}

# Ước lượng xác suất P(x_i = 1 | y = c) có làm mịn Laplace
p_ik = {}
symp_cols = X_Train_s.columns
for c in classes_s:
    sub = X_Train_s[Y_Train_s == c]
    N_c = len(sub)
    counts = sub.sum(axis=0)
    p_ik[c] = (counts + 1.0) / (N_c + 2.0)

def predict_bernoulli_scratch(X_val):
    preds = []
    for _, row in X_val.iterrows():
        scores = []
        for c in classes_s:
            s = np.log(priors_s[c])
            prob_vec = p_ik[c].values
            x_vec = row.values
            term = x_vec * prob_vec + (1 - x_vec) * (1 - prob_vec)
            s += np.sum(np.log(np.clip(term, 1e-12, 1.0)))
            scores.append(s)
        preds.append(classes_s[np.argmax(scores)])
    return np.array(preds)

y_pred_s_scratch = predict_bernoulli_scratch(X_Test_s)
print("Độ chính xác Bernoulli NB (Scratch):", accuracy_score(Y_Test_s, y_pred_s_scratch))
print("Khớp hoàn toàn với Sklearn:", np.all(y_pred_s_scratch == y_pred_s_sk))


Độ chính xác Bernoulli NB (Scratch): 0.485
Khớp hoàn toàn với Sklearn: True


## Bài tập (Động đất Nepal) - Bài 1: Multinomial Naïve Bayes trên 8 trường Categorical

In [15]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

# Đọc dữ liệu Động đất Nepal
train_values = pd.read_csv('train_values.csv')
train_labels = pd.read_csv('train_labels.csv')
y_nepal = train_labels['damage_grade'].values

cat_cols = [
    'land_surface_condition', 'foundation_type', 'roof_type',
    'ground_floor_type', 'other_floor_type', 'position',
    'plan_configuration', 'legal_ownership_status'
]

# Chia train:test tỉ lệ 80:20
X_tr_cat, X_te_cat, y_tr_nepal, y_te_nepal = train_test_split(
    train_values[cat_cols], y_nepal, test_size=0.2, random_state=42, stratify=y_nepal
)

# 1. Thư viện Scikit-learn (OneHotEncoder + MultinomialNB)
ohe_nepal = OneHotEncoder(sparse_output=True, handle_unknown='ignore')
X_tr_cat_ohe = ohe_nepal.fit_transform(X_tr_cat)
X_te_cat_ohe = ohe_nepal.transform(X_te_cat)

mnb_nepal = MultinomialNB(alpha=1.0)
mnb_nepal.fit(X_tr_cat_ohe, y_tr_nepal)
y_pred_cat_sk = mnb_nepal.predict(X_te_cat_ohe)
print("Độ chính xác Sklearn MultinomialNB:", accuracy_score(y_te_nepal, y_pred_cat_sk))

# 2. Phương pháp Multinomial NB tự cài đặt (From Scratch)
classes_nepal = np.unique(y_tr_nepal)
log_priors_nepal = np.array([np.log(np.mean(y_tr_nepal == c)) for c in classes_nepal])
num_features = X_tr_cat_ohe.shape[1]
log_probs_scratch = np.zeros((len(classes_nepal), num_features))

for i, c in enumerate(classes_nepal):
    sub = X_tr_cat_ohe[y_tr_nepal == c]
    counts = np.array(sub.sum(axis=0)).ravel()
    log_probs_scratch[i, :] = np.log((counts + 1.0) / (counts.sum() + num_features * 1.0))

scores_scratch = X_te_cat_ohe.dot(log_probs_scratch.T) + log_priors_nepal
y_pred_cat_scratch = classes_nepal[np.argmax(scores_scratch, axis=1)]
print("Độ chính xác Scratch MultinomialNB:", accuracy_score(y_te_nepal, y_pred_cat_scratch))


Độ chính xác Sklearn MultinomialNB: 0.5681395214980526
Độ chính xác Scratch MultinomialNB: 0.5681395214980526


## Bài tập (Động đất Nepal) - Bài 2: Bernoulli Naïve Bayes trên 22 trường Binary

In [16]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import BernoulliNB
from sklearn.metrics import accuracy_score

bin_cols = [c for c in train_values.columns if c.startswith('has_')]

# Chia train:test tỉ lệ 80:20
X_tr_bin, X_te_bin, _, _ = train_test_split(
    train_values[bin_cols], y_nepal, test_size=0.2, random_state=42, stratify=y_nepal
)

# 1. Thư viện Scikit-learn (BernoulliNB)
bnb_nepal = BernoulliNB(alpha=1.0)
bnb_nepal.fit(X_tr_bin, y_tr_nepal)
y_pred_bin_sk = bnb_nepal.predict(X_te_bin)
print("Độ chính xác Sklearn BernoulliNB:", accuracy_score(y_te_nepal, y_pred_bin_sk))

# 2. Phương pháp Bernoulli NB tự cài đặt (From Scratch)
classes_nepal = np.unique(y_tr_nepal)
log_priors = np.array([np.log(np.mean(y_tr_nepal == c)) for c in classes_nepal])

X_tr_mat = X_tr_bin.values
X_te_mat = X_te_bin.values
num_features = X_tr_mat.shape[1]
log_p = np.zeros((len(classes_nepal), num_features))
log_1_p = np.zeros((len(classes_nepal), num_features))

for i, c in enumerate(classes_nepal):
    sub = X_tr_mat[y_tr_nepal == c]
    N_c = len(sub)
    p_ik = (sub.sum(axis=0) + 1.0) / (N_c + 2.0)
    log_p[i, :] = np.log(p_ik)
    log_1_p[i, :] = np.log(1.0 - p_ik)

scores = np.zeros((len(X_te_mat), len(classes_nepal)))
for i in range(len(classes_nepal)):
    scores[:, i] = log_priors[i] + X_te_mat @ log_p[i] + (1 - X_te_mat) @ log_1_p[i]

y_pred_bin_scratch = classes_nepal[np.argmax(scores, axis=1)]
print("Độ chính xác Scratch BernoulliNB:", accuracy_score(y_te_nepal, y_pred_bin_scratch))


Độ chính xác Sklearn BernoulliNB: 0.5673528903896702
Độ chính xác Scratch BernoulliNB: 0.5673528903896702


## Ví dụ 2 (Phần C): Phân loại văn bản ẩm thực Bắc - Nam

In [17]:
import numpy as np
from sklearn.naive_bayes import MultinomialNB, BernoulliNB

# Tập dữ liệu training văn bản (D1..D4)
X_train_text = np.array([
    [2, 1, 1, 0, 0, 0, 0, 0, 0],  # D1: B
    [1, 1, 0, 1, 1, 0, 0, 0, 0],  # D2: B
    [0, 1, 0, 0, 1, 1, 0, 0, 0],  # D3: B
    [0, 1, 0, 0, 0, 0, 1, 1, 1]   # D4: N
])
y_train_text = np.array(['B', 'B', 'B', 'N'])

# Tập dữ liệu test (D5, D6)
X_test_text = np.array([
    [2, 0, 0, 1, 0, 0, 0, 1, 0],  # D5
    [0, 1, 0, 0, 0, 0, 0, 1, 1]   # D6
])

# 1. Multinomial Naive Bayes
clf_mnb = MultinomialNB()
clf_mnb.fit(X_train_text, y_train_text)
print("MultinomialNB dự đoán d5:", clf_mnb.predict(X_test_text[:1])[0])
print("MultinomialNB dự đoán d6:", clf_mnb.predict(X_test_text[1:2])[0])
print("Xác suất d5 (B, N):", clf_mnb.predict_proba(X_test_text[:1]))
print("Xác suất d6 (B, N):", clf_mnb.predict_proba(X_test_text[1:2]))

# 2. Bernoulli Naive Bayes
X_train_bin = (X_train_text > 0).astype(int)
X_test_bin = (X_test_text > 0).astype(int)

clf_bnb = BernoulliNB()
clf_bnb.fit(X_train_bin, y_train_text)
print("\nBernoulliNB dự đoán d5:", clf_bnb.predict(X_test_bin[:1])[0])
print("BernoulliNB dự đoán d6:", clf_bnb.predict(X_test_bin[1:2])[0])
print("Xác suất d5 (B, N):", clf_bnb.predict_proba(X_test_bin[:1]))
print("Xác suất d6 (B, N):", clf_bnb.predict_proba(X_test_bin[1:2]))


MultinomialNB dự đoán d5: B
MultinomialNB dự đoán d6: N
Xác suất d5 (B, N): [[0.89548823 0.10451177]]
Xác suất d6 (B, N): [[0.29175335 0.70824665]]

BernoulliNB dự đoán d5: B
BernoulliNB dự đoán d6: N
Xác suất d5 (B, N): [[0.76554295 0.23445705]]
Xác suất d6 (B, N): [[0.16948581 0.83051419]]


## Ví dụ 3 (Phần C): Phân loại Email Spam (Ling-Spam Dataset)

In [18]:
import numpy as np
from scipy.sparse import coo_matrix
from sklearn.naive_bayes import MultinomialNB, BernoulliNB
from sklearn.metrics import accuracy_score

path = 'ex6DataPrepared/'
train_data_fn = 'train-features.txt'
test_data_fn = 'test-features.txt'
train_label_fn = 'train-labels.txt'
test_label_fn = 'test-labels.txt'
nwords = 2500

def read_data(data_fn, label_fn):
    with open(path + label_fn, 'r') as f:
        labels = [int(x.strip()) for x in f if x.strip()]
    with open(path + data_fn, 'r') as f:
        lines = [x.strip() for x in f if x.strip()]
    dat = np.zeros((len(lines), 3), dtype=int)
    for i, line in enumerate(lines):
        p = line.split()
        dat[i, :] = [int(p[0]), int(p[1]), int(p[2])]
    data = coo_matrix((dat[:, 2], (dat[:, 0] - 1, dat[:, 1] - 1)), shape=(len(labels), nwords))
    return data, np.array(labels)

(train_data, train_label) = read_data(train_data_fn, train_label_fn)
(test_data, test_label) = read_data(test_data_fn, test_label_fn)

clf_spam = MultinomialNB()
clf_spam.fit(train_data, train_label)
y_pred_spam = clf_spam.predict(test_data)
print("Độ chính xác MultinomialNB (Ling-Spam):", accuracy_score(test_label, y_pred_spam))

clf_spam_b = BernoulliNB()
clf_spam_b.fit((train_data > 0).astype(int), train_label)
y_pred_spam_b = clf_spam_b.predict((test_data > 0).astype(int))
print("Độ chính xác BernoulliNB   (Ling-Spam):", accuracy_score(test_label, y_pred_spam_b))


Độ chính xác MultinomialNB (Ling-Spam): 0.9807692307692307
Độ chính xác BernoulliNB   (Ling-Spam): 0.8538461538461538


## Ví dụ 4 (Phần C): Phân loại đơn hàng trực tuyến bằng Scikit-learn

In [19]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score
import os

orders_fn = 'multinomial_nb_orders.csv' if os.path.exists('multinomial_nb_orders.csv') else 'multinomial_nb_orders (1).csv'
df_orders = pd.read_csv(orders_fn)
target_feature = 'cuisine'

Train_size = round(len(df_orders) * 0.7)
df_train = df_orders.iloc[:Train_size]
df_test = df_orders.iloc[Train_size:]

X_train = df_train.drop(columns=[target_feature])
y_train = df_train[target_feature]
X_test = df_test.drop(columns=[target_feature])
y_test = df_test[target_feature]

ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
X_train_ohe = ohe.fit_transform(X_train)
X_test_ohe = ohe.transform(X_test)

clf = MultinomialNB()
clf.fit(X_train_ohe, y_train)
y_pred = clf.predict(X_test_ohe)

print("Độ chính xác MultinomialNB trên tập Test (7:3):", accuracy_score(y_test, y_pred))


Độ chính xác MultinomialNB trên tập Test (7:3): 0.47555555555555556
